In [ ]:
import json
from pathlib import Path

import torch
from packaging.version import Version

OUTPUT_DIR = Path("/kaggle/working/out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available():
    raise RuntimeError("A Kaggle GPU runtime is required")
device_names = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
architectures = torch.cuda.get_arch_list()
print({"devices": device_names, "torch_architectures": architectures})
if len(device_names) != 2 or not all("t4" in name.lower() for name in device_names):
    raise RuntimeError(f"Expected two NVIDIA T4 devices, found: {device_names}")
if "sm_75" not in architectures:
    raise RuntimeError(f"The installed torch build lacks T4 sm_75 support: {architectures}")
torch_version = torch.__version__
if Version(torch_version.split("+")[0]) < Version("2.4"):
    raise RuntimeError(f"Kaggle torch {torch_version} is older than 2.4")
smoke_result = {"devices": device_names, "architectures": architectures, "torch": torch_version}
(OUTPUT_DIR / "smoke_result.json").write_text(
    json.dumps(smoke_result, indent=2) + "\n",
    encoding="utf-8",
)


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

INPUT_DIR = Path("/kaggle/input/criterion-lens-code")
CODE_DIR = Path("/kaggle/working/code")
requirements = (INPUT_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines()
requirements_without_torch = [
    line for line in requirements if not line.strip().lower().startswith("torch")
]
requirements_file = Path("/kaggle/working/requirements-no-torch.txt")
requirements_file.write_text("\n".join(requirements_without_torch) + "\n", encoding="utf-8")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)],
    check=True,
)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
shutil.copytree(INPUT_DIR, CODE_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"],
    cwd=CODE_DIR,
    check=True,
)
print(f"Installed allowlisted project source from {INPUT_DIR}")


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

manifest_path = CODE_DIR / "data" / "splits" / "split_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
subprocess.run(
    [sys.executable, "-u", "scripts/make_split.py", "--revision", manifest["dataset_revision"]],
    cwd=CODE_DIR,
    check=True,
)
jobs = (
    ("qwen25_1p5b_lora", "configs/train_qwen25_1p5b.yaml", "0"),
    ("qwen25_0p5b_lora", "configs/train_qwen25_0p5b.yaml", "1"),
)
processes = []
log_paths = {}
log_handles = []
for run_name, config_path, gpu_id in jobs:
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = gpu_id
    environment["PYTHONUNBUFFERED"] = "1"
    log_path = OUTPUT_DIR / f"log_{run_name}.txt"
    log_handle = log_path.open("w", encoding="utf-8")
    log_handles.append(log_handle)
    process = subprocess.Popen(
        [sys.executable, "-u", "scripts/train.py", "--config", config_path, "--run_name", run_name],
        cwd=CODE_DIR,
        env=environment,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )
    processes.append((run_name, process))
failures = []
for run_name, process in processes:
    return_code = process.wait()
    if return_code:
        failures.append((run_name, return_code))
for log_handle in log_handles:
    log_handle.close()
if failures:
    for run_name, return_code in failures:
        log_path = OUTPUT_DIR / f"log_{run_name}.txt"
        print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-100:]))
    raise RuntimeError(f"Training jobs failed: {failures}")
print("Both target and draft training jobs completed successfully")


In [ ]:
import shutil
import zipfile
from pathlib import Path

artifact_root = OUTPUT_DIR / "training_artifacts"
for run_name, _, _ in jobs:
    run_dir = CODE_DIR / "outputs" / run_name
    adapter_dir = artifact_root / "adapters" / run_name
    adapter_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(
        run_dir,
        adapter_dir,
        ignore=shutil.ignore_patterns("checkpoint-*"),
    )
    result_dir = artifact_root / "results" / "train" / run_name
    result_dir.mkdir(parents=True, exist_ok=True)
    filenames = (
        "log_history.json",
        "train_config.json",
        "trainable_params.json",
        "truncation_log.json",
    )
    for filename in filenames:
        source = run_dir / filename
        if source.is_file():
            shutil.copy2(source, result_dir / filename)
for log_path in OUTPUT_DIR.glob("log_*.txt"):
    shutil.copy2(log_path, artifact_root / "results" / "train" / log_path.name)
archive_path = Path("/kaggle/working/training_artifacts.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in artifact_root.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(artifact_root))
shutil.move(str(archive_path), str(OUTPUT_DIR / "training_artifacts.zip"))
print(f"Packaged training outputs at {OUTPUT_DIR / 'training_artifacts.zip'}")
